# MuRIL training on Colab or Kaggle

Run the cells in order. Set the `HF_TOKEN` secret and, optionally, `HF_MODEL_ID` in the notebook environment to publish the trained model.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected; training will use CPU and may be slow.")

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/neural-strom-hi02")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/MayankLumbhani/neural-strom-hi02.git", str(repo_path)],
        check=True,
    )
os.chdir(repo_path)

In [ ]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[training]"],
    check=True,
)

In [ ]:
from google.colab import files

uploaded = files.upload()
required_files = {"train.csv", "validation.csv"}
missing_files = required_files.difference(uploaded)
if missing_files:
    raise ValueError(f"Upload both required files: {', '.join(sorted(missing_files))}")
data_dir = repo_path / "ml" / "data" / "processed"
data_dir.mkdir(parents=True, exist_ok=True)
for filename in required_files:
    (data_dir / filename).write_bytes(uploaded[filename])
print("Training and validation CSVs are in ml/data/processed/.")

In [ ]:
import os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN loaded from notebook secrets; its value is not displayed.")

In [ ]:
import os
import subprocess
import sys

command = [sys.executable, "-m", "ml.src.train_muril"]
hub_model_id = os.environ.get("HF_MODEL_ID")
if hub_model_id:
    command.extend(["--push_to_hub", "--hub_model_id", hub_model_id])
subprocess.run(command, check=True)

In [ ]:
from google.colab import files

metrics_path = repo_path / "ml" / "artifacts" / "muril_metrics.json"
print(metrics_path.read_text(encoding="utf-8"))
files.download(str(metrics_path))